In [4]:
# Ejemplo End-to-End 1: Fine-Tuning GPT-2 para Forecasting de Demanda por Categoría
# CORRECCIÓN DEL ERROR MPS: Forzamos el modelo e inputs a CPU para evitar problemas con MPS device en Apple Silicon.
# MPS (Metal Performance Shaders) tiene bugs conocidos con generation en algunos modelos PyTorch.
# Esto asegura compatibilidad en cualquier entorno (CPU es suficiente para GPT-2 pequeño).
# Requiere: pip install transformers datasets torch accelerate pandas

import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch

# Paso 1: Cargar y preparar el dataset
url = 'https://raw.githubusercontent.com/narencastellon/Serie-de-tiempo-con-Machine-Learning/refs/heads/main/Data/Supplement_Sales_Weekly_Expanded.csv'
df = pd.read_csv(url)
df

,Date,Product Name,Category,Units Sold,Price,Revenue,Discount,Units Returned,Location,Platform
0,2020-01-06,Whey Protein,Protein,143,31.98,4573.14,0.03,2,Canada,Walmart
1,2020-01-06,Vitamin C,Vitamin,139,42.51,5908.89,0.04,0,UK,Amazon
2,2020-01-06,Fish Oil,Omega,161,12.91,2078.51,0.25,0,Canada,Amazon
3,2020-01-06,Multivitamin,Vitamin,140,16.07,2249.80,0.08,0,Canada,Walmart
4,2020-01-06,Pre-Workout,Performance,157,35.47,5568.79,0.25,3,Canada,iHerb
...,...,...,...,...,...,...,...,...,...,...
4379,2025-03-31,Melatonin,Sleep Aid,160,47.79,7646.40,0.21,1,USA,iHerb
4380,2025-03-31,Biotin,Vitamin,154,38.12,5870.48,0.22,1,UK,Walmart
4381,2025-03-31,Green Tea Extract,Fat Burner,139,20.40,2835.60,0.12,3,USA,iHerb
4382,2025-03-31,Iron Supplement,Mineral,154,18.31,2819.74,0.23,2,Canada,Amazon


In [5]:
# Convertir 'Date' a datetime y ordenar
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Category', 'Date'])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4384 entries, 5 to 4380
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            4384 non-null   datetime64[ns]
 1   Product Name    4384 non-null   object        
 2   Category        4384 non-null   object        
 3   Units Sold      4384 non-null   int64         
 4   Price           4384 non-null   float64       
 5   Revenue         4384 non-null   float64       
 6   Discount        4384 non-null   float64       
 7   Units Returned  4384 non-null   int64         
 8   Location        4384 non-null   object        
 9   Platform        4384 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 376.8+ KB


In [ ]:
# Agrupar demanda semanal por Category (suma Units Sold)
df_category = df.groupby(['Category', 'Date'])['Units Sold'].sum().reset_index()
df_category.rename(columns={'Units Sold': 'Demand'}, inplace=True)

print("Dataset preparado por categoría (primeras filas):")
print(df_category.head(10))
print("Categorías únicas:", df_category['Category'].unique())

In [7]:
df_category

,Category,Date,Demand
0,Amino Acid,2020-01-06,154
1,Amino Acid,2020-01-13,160
2,Amino Acid,2020-01-20,145
3,Amino Acid,2020-01-27,136
4,Amino Acid,2020-02-03,126
...,...,...,...
2735,Vitamin,2025-03-03,415
2736,Vitamin,2025-03-10,468
2737,Vitamin,2025-03-17,375
2738,Vitamin,2025-03-24,444


In [8]:
# Paso 2: Crear prompts de texto para fine-tuning
def create_prompts(group, context_length=20):
    demand = group['Demand'].values
    prompts = []
    for i in range(context_length, len(demand)):
        context = ', '.join(map(str, demand[i-context_length:i].astype(int)))
        target = int(demand[i])
        prompt = f"Category: {group['Category'].iloc[0]} | Historical weekly demand: {context} | Next week demand:"
        completion = f" {target}"
        prompts.append(prompt + completion)
    return prompts

all_prompts = []
for category, group in df_category.groupby('Category'):
    prompts = create_prompts(group)
    all_prompts.extend(prompts)

print(f"Generados {len(all_prompts)} prompts para fine-tuning.")

Generados 2540 prompts para fine-tuning.


In [9]:
all_prompts

['Category: Amino Acid | Historical weekly demand: 154, 160, 145, 136, 126, 134, 152, 147, 169, 168, 151, 142, 140, 153, 158, 147, 139, 146, 128, 157 | Next week demand: 136',
 'Category: Amino Acid | Historical weekly demand: 160, 145, 136, 126, 134, 152, 147, 169, 168, 151, 142, 140, 153, 158, 147, 139, 146, 128, 157, 136 | Next week demand: 154',
 'Category: Amino Acid | Historical weekly demand: 145, 136, 126, 134, 152, 147, 169, 168, 151, 142, 140, 153, 158, 147, 139, 146, 128, 157, 136, 154 | Next week demand: 149',
 'Category: Amino Acid | Historical weekly demand: 136, 126, 134, 152, 147, 169, 168, 151, 142, 140, 153, 158, 147, 139, 146, 128, 157, 136, 154, 149 | Next week demand: 129',
 'Category: Amino Acid | Historical weekly demand: 126, 134, 152, 147, 169, 168, 151, 142, 140, 153, 158, 147, 139, 146, 128, 157, 136, 154, 149, 129 | Next week demand: 174',
 'Category: Amino Acid | Historical weekly demand: 134, 152, 147, 169, 168, 151, 142, 140, 153, 158, 147, 139, 146, 128,

In [10]:
# Paso 3: Tokenizar con labels para causal LM
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    outputs = tokenizer(examples['text'], truncation=True, max_length=256, padding='max_length')
    outputs['labels'] = outputs['input_ids'].copy()  # Labels para calcular loss
    return outputs

data_dict = {"text": all_prompts}
dataset = Dataset.from_dict(data_dict)
tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 98ff169e-b1d8-4767-aca6-67fb94ac5662)')' thrown while requesting HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Map: 100%|██████████| 2540/2540 [00:00<00:00, 4287.97 examples/s]


In [14]:
tokenized_dataset

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2540
})

In [15]:
# Paso 4: Fine-tuning de GPT-2 (en CPU para evitar MPS errors)
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.to('cpu')  # Forzar CPU para compatibilidad universal

training_args = TrainingArguments(
    output_dir='./gpt2-finetuned-demand',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=200,
    fp16=False  # Desactivar fp16 si no hay GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Iniciando fine-tuning de GPT-2...")
trainer.train()

model.save_pretrained('./gpt2-finetuned-demand')
tokenizer.save_pretrained('./gpt2-finetuned-demand')
print("Fine-tuning de GPT-2 completado.")

Iniciando fine-tuning de GPT-2...


/opt/miniconda3/envs/myenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,20.042200
100,1.843500
150,1.670800
200,1.662600
250,1.636400
300,1.630500
350,1.603500
400,1.604700
450,1.591700
500,1.577900


/opt/miniconda3/envs/myenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/miniconda3/envs/myenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Fine-tuning de GPT-2 completado.


In [19]:


# Paso 5: Inferencia (Forecast) con CPU explícito
def forecast_gpt2(category_group, model, tokenizer, context_length = 50, horizon = 6):
    model.to('cpu')  # Asegurar CPU
    demand = category_group['Demand'].values[-context_length:].tolist()
    category = category_group['Category'].iloc[0]
    input_text = f"Category: {category} | Historical weekly demand: {', '.join(map(str, map(int, demand)))} | Next week demand:"
    
    forecast = []
    for _ in range(horizon):
        inputs = tokenizer(input_text, return_tensors="pt")
        inputs = {k: v.to('cpu') for k, v in inputs.items()}  # Mover inputs a CPU
        outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.7, do_sample=True)
        new_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        try:
            predicted = int(new_text.split("Next week demand:")[-1].strip().split()[0])
        except:
            predicted = int(np.mean(demand[-5:]))
        forecast.append(predicted)
        demand.append(predicted)
        input_text = f"Category: {category} | Historical weekly demand: {', '.join(map(str, map(int, demand[-context_length:])))} | Next week demand:"
    
    return forecast

example_category = df_category[df_category['Category'] == df_category['Category'].unique()[0]]
forecast = forecast_gpt2(example_category, model, tokenizer)
print(f"Forecast GPT-2 para {example_category['Category'].iloc[0]}: {forecast}")



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Forecast GPT-2 para Amino Acid: [157, 142, 138, 153, 141, 152]


In [17]:
example_category

,Category,Date,Demand
0,Amino Acid,2020-01-06,154
1,Amino Acid,2020-01-13,160
2,Amino Acid,2020-01-20,145
3,Amino Acid,2020-01-27,136
4,Amino Acid,2020-02-03,126
...,...,...,...
269,Amino Acid,2025-03-03,159
270,Amino Acid,2025-03-10,148
271,Amino Acid,2025-03-17,163
272,Amino Acid,2025-03-24,143


In [20]:
# === Ejemplo End-to-End 2: Fine-Tuning facebook/opt-125m ===
# Similar corrección: Forzar CPU y mover inputs

from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer_opt = AutoTokenizer.from_pretrained("facebook/opt-125m")
model_opt = AutoModelForCausalLM.from_pretrained("facebook/opt-125m")
model_opt.to('cpu')  # Forzar CPU

def tokenize_opt(examples):
    outputs = tokenizer_opt(examples['text'], truncation=True, max_length=256, padding='max_length')
    outputs['labels'] = outputs['input_ids'].copy()
    return outputs

tokenized_opt = dataset.map(tokenize_opt, batched=True)

training_args_opt = TrainingArguments(
    output_dir='./opt125m-finetuned-demand',
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=False,  # Desactivar para CPU
    logging_steps=50
)

trainer_opt = Trainer(
    model=model_opt,
    args=training_args_opt,
    train_dataset=tokenized_opt,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer_opt, mlm=False)
)

print("Iniciando fine-tuning de OPT-125m...")
trainer_opt.train()

model_opt.save_pretrained('./opt125m-finetuned-demand')
tokenizer_opt.save_pretrained('./opt125m-finetuned-demand')
print("Fine-tuning OPT-125m completado.")

# Inferencia OPT con CPU
def forecast_opt(category_group, model, tokenizer, context_length=20, horizon=4):
    model.to('cpu')
    demand = category_group['Demand'].values[-context_length:].tolist()
    category = category_group['Category'].iloc[0]
    input_text = f"Category: {category} | Historical weekly demand: {', '.join(map(str, map(int, demand)))} | Next week demand:"
    
    forecast = []
    for _ in range(horizon):
        inputs = tokenizer(input_text, return_tensors="pt")
        inputs = {k: v.to('cpu') for k, v in inputs.items()}
        outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.7, do_sample=True)
        new_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        try:
            predicted = int(new_text.split("Next week demand:")[-1].strip().split()[0])
        except:
            predicted = int(np.mean(demand[-5:]))
        forecast.append(predicted)
        demand.append(predicted)
        input_text = f"Category: {category} | Historical weekly demand: {', '.join(map(str, map(int, demand[-context_length:])))} | Next week demand:"
    
    return forecast

forecast_opt_example = forecast_opt(example_category, model_opt, tokenizer_opt)
print(f"Forecast OPT-125m para {example_category['Category'].iloc[0]}: {forecast_opt_example}")

print("\n¡Código corregido para MPS/CPU! Ahora generation funciona sin RuntimeError al forzar CPU y mover tensors.")

'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: 8cf81cd6-4010-49de-aa35-bbe3a247f3a4)')' thrown while requesting HEAD https://huggingface.co/facebook/opt-125m/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Map: 100%|██████████| 2540/2540 [00:00<00:00, 7775.03 examples/s]


Iniciando fine-tuning de OPT-125m...


/opt/miniconda3/envs/myenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,2.350900
100,1.619200
150,1.581100
200,1.531000
250,1.466800
300,1.388400


Fine-tuning OPT-125m completado.
Forecast OPT-125m para Amino Acid: [153, 152, 152, 150]

¡Código corregido para MPS/CPU! Ahora generation funciona sin RuntimeError al forzar CPU y mover tensors.


In [21]:
import pandas as pd
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
from datetime import timedelta

# Paso 1: Cargar y preparar el dataset
url = 'https://raw.githubusercontent.com/narencastellon/Serie-de-tiempo-con-Machine-Learning/refs/heads/main/Data/Supplement_Sales_Weekly_Expanded.csv'
df = pd.read_csv(url)

# Convertir 'Date' a datetime y ordenar
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(['Category', 'Date'])

# Agrupar demanda semanal por Category (suma Units Sold)
df_category = df.groupby(['Category', 'Date'])['Units Sold'].sum().reset_index()
df_category.rename(columns={'Units Sold': 'Demand'}, inplace=True)

print("Dataset preparado por categoría (primeras filas):")
print(df_category.head(10))
print("Categorías únicas:", df_category['Category'].unique())

# Paso 2: Cargar el modelo fine-tuned (asumiendo que ya ejecutaste el fine-tuning y guardaste)
# Si no lo tienes, ejecuta el código de fine-tuning primero
model_path = './gpt2-finetuned-demand'  # Ruta donde guardaste el modelo
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_path)
model.eval()  # Modo evaluación
model.to('cpu')  # Forzar CPU para evitar errores MPS/GPU

print("Modelo GPT-2 fine-tuned cargado exitosamente.")

# Paso 3: Función de forecast (autoregresivo)
def forecast_gpt2(category_group, model, tokenizer, context_length=20, horizon=4):
    """
    Genera forecast autoregresivo para 'horizon' semanas.
    - category_group: DataFrame filtrado por categoría.
    - model/tokenizer: Modelo fine-tuned.
    - context_length: Semanas históricas usadas como contexto.
    - horizon: Semanas a pronosticar.
    Retorna lista de valores pronosticados.
    """
    # Asegurar contexto suficiente
    if len(category_group) < context_length:
        context_length = len(category_group)
    
    demand = category_group['Demand'].values[-context_length:].tolist()
    category = category_group['Category'].iloc[0]
    input_text = f"Category: {category} | Historical weekly demand: {', '.join(map(str, map(int, demand)))} | Next week demand:"
    
    forecast = []
    current_context = demand.copy()
    
    for _ in range(horizon):
        inputs = tokenizer(input_text, return_tensors="pt")
        inputs = {k: v.to('cpu') for k, v in inputs.items()}  # CPU seguro
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        
        new_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        try:
            predicted = int(new_text.split("Next week demand:")[-1].strip().split()[0])
        except:
            predicted = int(np.mean(current_context[-5:]))  # Fallback: media reciente
        predicted = max(predicted, 0)  # No negativos
        
        forecast.append(predicted)
        current_context.append(predicted)
        
        # Actualizar contexto para próximo paso
        input_text = f"Category: {category} | Historical weekly demand: {', '.join(map(str, map(int, current_context[-context_length:])))} | Next week demand:"
    
    return forecast

# Paso 4: Generar forecast para TODAS las categorías
horizon = 4  # Pronosticar 4 semanas futuras (ajustable)
context_length = 20  # Contexto histórico

forecast_records = []  # Lista para almacenar resultados

print("Generando forecast para todas las categorías...")

for category, group in df_category.groupby('Category'):
    print(f"Procesando categoría: {category}")
    
    # Obtener forecast
    forecast_values = forecast_gpt2(group, model, tokenizer, context_length=context_length, horizon=horizon)
    
    # Crear fechas futuras (comunes para todas, basadas en la última fecha global)
    last_date = df_category['Date'].max()
    future_dates = pd.date_range(start=last_date + timedelta(weeks=1), periods=horizon, freq='W')
    
    # Agregar al registro
    for i in range(horizon):
        forecast_records.append({
            'Category': category,
            'Date': future_dates[i],
            'Forecast_Demand': forecast_values[i]
        })

# Crear DataFrame final con todos los forecasts
forecast_df = pd.DataFrame(forecast_records)

# Mostrar resultados
print("\nForecast generado para todas las categorías (primeras 20 filas):")
print(forecast_df.head(20))

print("\nResumen por categoría:")
print(forecast_df.groupby('Category')['Forecast_Demand'].sum().reset_index(name='Total_Forecast_4_Weeks'))

# Guardar el DataFrame como CSV
forecast_df.to_csv('forecast_all_categories_gpt2.csv', index=False)
print("\n¡Forecast completo guardado en 'forecast_all_categories_gpt2.csv'! Contiene pronósticos para todas las categorías.")

Dataset preparado por categoría (primeras filas):
     Category       Date  Demand
0  Amino Acid 2020-01-06     154
1  Amino Acid 2020-01-13     160
2  Amino Acid 2020-01-20     145
3  Amino Acid 2020-01-27     136
4  Amino Acid 2020-02-03     126
5  Amino Acid 2020-02-10     134
6  Amino Acid 2020-02-17     152
7  Amino Acid 2020-02-24     147
8  Amino Acid 2020-03-02     169
9  Amino Acid 2020-03-09     168
Categorías únicas: ['Amino Acid' 'Fat Burner' 'Herbal' 'Hydration' 'Mineral' 'Omega'
 'Performance' 'Protein' 'Sleep Aid' 'Vitamin']
Modelo GPT-2 fine-tuned cargado exitosamente.
Generando forecast para todas las categorías...
Procesando categoría: Amino Acid
Procesando categoría: Fat Burner
Procesando categoría: Herbal
Procesando categoría: Hydration
Procesando categoría: Mineral
Procesando categoría: Omega
Procesando categoría: Performance
Procesando categoría: Protein
Procesando categoría: Sleep Aid
Procesando categoría: Vitamin

Forecast generado para todas las categorías (pri

In [22]:
forecast_df

,Category,Date,Forecast_Demand
0,Amino Acid,2025-04-13,156
1,Amino Acid,2025-04-20,143
2,Amino Acid,2025-04-27,152
3,Amino Acid,2025-05-04,150
4,Fat Burner,2025-04-13,144
5,Fat Burner,2025-04-20,146
6,Fat Burner,2025-04-27,144
7,Fat Burner,2025-05-04,167
8,Herbal,2025-04-13,147
9,Herbal,2025-04-20,151
